# HarnessChandelier

*Harness the Chandelier — connect to the center of your conversation.*  

**Topic Drift Tracking for Long-Running Agent Conversations**

In real-world AI agent interactions, users naturally drift across multiple topics —
then return to what they originally wanted.

HarnessChandelier doesn't try to prevent drift.
It **tracks it** — and finds the topic the user kept coming back to.

Like a chandelier at the center of a hall, the dominant topic stays fixed
no matter how much the conversation moves around it.

Built on:
- **BERTopic** + cuML UMAP/HDBSCAN — GPU-accelerated topic extraction
- **cuGraph PageRank** — topic importance ranking
- **Temporal edge weighting** — time-aware topic transition graph

Built for **Harness Engineering** workflows where conversations drift across topics.

This example demonstrates a realistic scenario where a user frequently switches between unrelated topics throughout a long conversation — a common pattern observed in real-world AI assistant interactions

In [1]:
from harness_chandelier import HarnessChandelier

messages = [
    # A - trip planning
    "I want to plan a trip to Japan next spring. Where should I start?",
    "Should I visit Tokyo first or Kyoto? I have 10 days total.",
    "What's the best way to get a JR Pass? Is it worth it?",
    
    # B - coding ..all of a sudden
    "Hey, totally different topic. I'm getting a TypeError in my Python code. Can you help?",
    "The error says 'NoneType object is not subscriptable'. What does that mean?",
    "Okay I fixed it. Thanks. It was a simple null check issue.",
    
    # C - life advice ..again, out of nowhere
    "Can I ask you something personal? I'm thinking about quitting my job.",
    "I've been at this company for 7 years but I feel stuck. No growth.",
    "My salary is okay but I'm not happy. Is it worth leaving for uncertainty?",
    "I have a family to support. That makes it harder to just quit.",
    
    # A - back to trip
    "Sorry, back to Japan. How much money should I budget for 10 days?",
    "Is Osaka worth visiting or should I skip it for more time in Kyoto?",
    "What about the bullet train? Is it as fast as they say?",
    
    # B - coding again
    "New coding problem. I'm trying to scrape a website but getting blocked.",
    "Should I use Selenium or BeautifulSoup for this?",
    "The website has JavaScript rendering. Does that change things?",
    "Got it working with Selenium. But it's really slow. Any tips?",
    
    # C - life advice again
    "Back to the job thing. I got a recruiter message today on LinkedIn.",
    "The new role pays 30% more but it's a startup. Risk vs reward?",
    "My wife thinks I should take it. My parents think I should stay safe.",
    "What would you do in my situation?",
    
    # A - back to trip ..again
    "Japan again - I found cheap flights for April. Should I book now?",
    "Cherry blossom season is April right? Will it overlap with my trip?",
    "Hotels in Tokyo are expensive. Should I try Airbnb instead?",
    "What neighborhoods in Tokyo are best for first timers?",
    
    # C - life advice again..
    "I applied to the startup job. Just did the first interview.",
    "They asked me where I see myself in 5 years. I had no idea what to say.",
    "I think I bombed it. Feel really discouraged right now.",
    "Maybe I should just stay where I am. At least it's stable.",
    
    # B - coding..lol
    "Different coding question. I'm learning React. Where should I start?",
    "Is it better to learn class components or just go straight to hooks?",
    "I built my first component but props aren't passing correctly.",
    "Never mind I figured it out. Missing curly braces. Classic.",
    
    # A -  back to trip ..again 2
    "Booked the Japan flights! April 3rd to April 13th.",
    "Now I need to figure out accommodation. 10 nights budget options?",
    "Should I do a ryokan experience even if it's expensive?",
    "I want to see Mount Fuji. Day trip from Tokyo or stay nearby?",
    "What about food allergies? I'm vegetarian. Is Japan hard for that?",
    
    # C - life advice again..for the last time I promise
    "I got a second interview at the startup! Maybe I didn't bomb it.",
    "They want me to do a technical presentation. I'm nervous.",
    "If I get this job my whole life changes. Is that scary or exciting?",
    "I keep going back and forth. Some days I want it, some days I don't.",
    
    # B - coding
    "My React app is almost done but deployment is confusing me.",
    "Vercel vs Netlify for a beginner? Which is easier?",
    "Deployed! But the environment variables aren't working in production.",
    
    # A - back to trip !!!!!
    "One month until Japan! Still haven't planned the actual itinerary.",
    "Can you give me a day by day plan for 10 days in Japan?",
    "Day 1 Tokyo, Day 2 Tokyo, Day 3 Kyoto... is that realistic?",
    
    # C - life advice again..last few messages
    "I got the job offer. They want an answer by Friday.",
    "I'm terrified. This is the biggest decision of my life.",
    "I think I'm going to take it. Wish me luck.",
    
    # A -  yes. really back to trip
    "Two weeks until Japan! I'm so excited. Any last minute tips?",
    "Should I get travel insurance? Is it worth it for Japan?",
    "Packing list for Japan in April? What should I not forget?",
    "I leave tomorrow. Any final advice?",
]

In [2]:
from datetime import datetime

real_timestamps = [
    # A - Travel Planning (January 10, morning)
    datetime(2026, 1, 10, 9, 0, 5),    # 0: Japan trip idea
    datetime(2026, 1, 10, 9, 1, 30),   # 1: Tokyo vs Kyoto
    datetime(2026, 1, 10, 9, 3, 45),   # 2: JR Pass

    # B - Coding (suddenly, 2 hours later)
    datetime(2026, 1, 10, 11, 0, 0),   # 3: TypeError
    datetime(2026, 1, 10, 11, 2, 15),  # 4: NoneType error
    datetime(2026, 1, 10, 11, 15, 0),  # 5: fixed

    # C - Life Advice (next day)
    datetime(2026, 1, 11, 9, 30, 0),   # 6: thinking about quitting
    datetime(2026, 1, 11, 9, 32, 0),   # 7: 7 years at same company
    datetime(2026, 1, 11, 9, 35, 0),   # 8: not happy
    datetime(2026, 1, 11, 9, 40, 0),   # 9: family to support

    # A - Travel (back, after lunch)
    datetime(2026, 1, 11, 13, 0, 0),   # 10: Japan budget
    datetime(2026, 1, 11, 13, 3, 0),   # 11: Osaka
    datetime(2026, 1, 11, 13, 5, 0),   # 12: bullet train

    # B - Coding (evening)
    datetime(2026, 1, 11, 20, 0, 0),   # 13: web scraping
    datetime(2026, 1, 11, 20, 3, 0),   # 14: Selenium vs BS
    datetime(2026, 1, 11, 20, 8, 0),   # 15: JS rendering
    datetime(2026, 1, 11, 20, 30, 0),  # 16: Selenium too slow

    # C - Life Advice (back, next morning)
    datetime(2026, 1, 12, 8, 0, 0),    # 17: LinkedIn recruiter
    datetime(2026, 1, 12, 8, 3, 0),    # 18: 30% raise startup
    datetime(2026, 1, 12, 8, 6, 0),    # 19: wife vs parents
    datetime(2026, 1, 12, 8, 10, 0),   # 20: what would you do?

    # A - Travel (back, one week later)
    datetime(2026, 1, 19, 10, 0, 0),   # 21: cheap flights found
    datetime(2026, 1, 19, 10, 2, 0),   # 22: cherry blossom season
    datetime(2026, 1, 19, 10, 5, 0),   # 23: hotel vs Airbnb
    datetime(2026, 1, 19, 10, 8, 0),   # 24: Tokyo neighborhoods

    # C - Life Advice (few days later)
    datetime(2026, 1, 22, 14, 0, 0),   # 25: first interview
    datetime(2026, 1, 22, 14, 5, 0),   # 26: where do you see yourself
    datetime(2026, 1, 22, 14, 10, 0),  # 27: think I bombed it
    datetime(2026, 1, 22, 14, 15, 0),  # 28: maybe just stay

    # B - Coding (evening)
    datetime(2026, 1, 22, 21, 0, 0),   # 29: learning React
    datetime(2026, 1, 22, 21, 5, 0),   # 30: hooks vs class
    datetime(2026, 1, 22, 21, 20, 0),  # 31: props not passing
    datetime(2026, 1, 22, 21, 45, 0),  # 32: fixed, curly braces

    # A - Travel (February)
    datetime(2026, 2, 1, 9, 0, 0),     # 33: flights booked!
    datetime(2026, 2, 1, 9, 5, 0),     # 34: 10 nights accommodation
    datetime(2026, 2, 1, 9, 10, 0),    # 35: ryokan experience
    datetime(2026, 2, 1, 9, 15, 0),    # 36: Mount Fuji
    datetime(2026, 2, 1, 9, 20, 0),    # 37: vegetarian in Japan

    # C - Life Advice
    datetime(2026, 2, 5, 10, 0, 0),    # 38: second interview!
    datetime(2026, 2, 5, 10, 5, 0),    # 39: technical presentation
    datetime(2026, 2, 5, 10, 10, 0),   # 40: life is changing
    datetime(2026, 2, 5, 10, 15, 0),   # 41: back and forth

    # B - Coding
    datetime(2026, 2, 10, 20, 0, 0),   # 42: React deployment
    datetime(2026, 2, 10, 20, 10, 0),  # 43: Vercel vs Netlify
    datetime(2026, 2, 10, 20, 45, 0),  # 44: env variables broken

    # A - Travel (March)
    datetime(2026, 3, 1, 9, 0, 0),     # 45: one month left!
    datetime(2026, 3, 1, 9, 5, 0),     # 46: day by day itinerary
    datetime(2026, 3, 1, 9, 10, 0),    # 47: Tokyo Kyoto realistic?

    # C - Life Advice
    datetime(2026, 3, 15, 18, 0, 0),   # 48: got the offer!
    datetime(2026, 3, 15, 18, 5, 0),   # 49: answer by Friday
    datetime(2026, 3, 15, 18, 10, 0),  # 50: going to take it

    # A - Travel (April, final stretch)
    datetime(2026, 4, 1, 8, 0, 0),     # 51: two weeks left!
    datetime(2026, 4, 1, 8, 5, 0),     # 52: travel insurance
    datetime(2026, 4, 1, 8, 10, 0),    # 53: packing list
    datetime(2026, 4, 2, 20, 0, 0),    # 54: leaving tomorrow!
]

In [3]:
ranker = HarnessChandelier(
    weights={"delta_time": 0.2}
)

result = ranker.fit(messages, timestamps=real_timestamps)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(f"Main Topic: {result.main_topic}")
print(f"Main Topic Keywords: {result.main_topic_keywords}")  #
print()
print("=== PageRank (Topic Importance) ===")
print(result.pagerank)

Main Topic: 0
Main Topic Keywords: ['got', 'job', 'interview', 'years', 'startup']

=== PageRank (Topic Importance) ===
    vertex  pagerank
0        0  0.179002
1        3  0.111841
2        6  0.105844
3        5  0.097809
4        1  0.095309
5        2  0.077117
6        7  0.069491
7        4  0.064226
8        9  0.063474
9        8  0.051208
10      11  0.049376
11      10  0.035301


In [5]:
from collections import Counter

print("=== Topic Distribution ===")
counter = Counter(result.topic_labels)
for topic, count in sorted(counter.items()):
    print(f"Topic {topic}: {count} count")

print()
print("=== Topic per Message ===")
for i, (msg, topic) in enumerate(zip(messages, result.topic_labels)):
    print(f"[{i:02d}] Topic {topic}: {msg[:55]}...")

=== Topic Distribution ===
Topic -1: 2 count
Topic 0: 7 count
Topic 1: 7 count
Topic 2: 6 count
Topic 3: 5 count
Topic 4: 4 count
Topic 5: 4 count
Topic 6: 4 count
Topic 7: 4 count
Topic 8: 3 count
Topic 9: 3 count
Topic 10: 3 count
Topic 11: 3 count

=== Topic per Message ===
[00] Topic -1: I want to plan a trip to Japan next spring. Where shoul...
[01] Topic 7: Should I visit Tokyo first or Kyoto? I have 10 days tot...
[02] Topic 4: What's the best way to get a JR Pass? Is it worth it?...
[03] Topic 8: Hey, totally different topic. I'm getting a TypeError i...
[04] Topic 8: The error says 'NoneType object is not subscriptable'. ...
[05] Topic 3: Okay I fixed it. Thanks. It was a simple null check iss...
[06] Topic 6: Can I ask you something personal? I'm thinking about qu...
[07] Topic 0: I've been at this company for 7 years but I feel stuck....
[08] Topic 10: My salary is okay but I'm not happy. Is it worth leavin...
[09] Topic 6: I have a family to support. That makes it harder to

### Short, context-free messages tend to become outliers (Topic -1) in BERTopic, which HarnessTopicRanker naturally assigns the lowest PageRank — effectively filtering noise from the dominant topic detection.

In [6]:
print("=== Topic -1 (Outlier) messages ===")
for i, (msg, topic) in enumerate(zip(messages, result.topic_labels)):
    if topic == -1:
        print(f"[{i:02d}] {msg}")

=== Topic -1 (Outlier) messages ===
[00] I want to plan a trip to Japan next spring. Where should I start?
[24] What neighborhoods in Tokyo are best for first timers?


In [7]:
# Final Summary
print("=== Dominant Topic Analysis ===")
main_topic_messages = [
    (i, msg) for i, (msg, topic) 
    in enumerate(zip(messages, result.topic_labels)) 
    if topic == result.main_topic
]

print(f"Main Topic: {result.main_topic} (PageRank: {result.pagerank.iloc[0]['pagerank']:.4f})")
print(f"Appears in {len(main_topic_messages)} out of {len(messages)} messages")
print()
print("Messages classified as Main Topic:")
for i, msg in main_topic_messages:
    print(f"  [{i:02d}] {msg[:70]}...")

=== Dominant Topic Analysis ===
Main Topic: 0 (PageRank: 0.1790)
Appears in 7 out of 55 messages

Messages classified as Main Topic:
  [07] I've been at this company for 7 years but I feel stuck. No growth....
  [17] Back to the job thing. I got a recruiter message today on LinkedIn....
  [25] I applied to the startup job. Just did the first interview....
  [26] They asked me where I see myself in 5 years. I had no idea what to say...
  [38] I got a second interview at the startup! Maybe I didn't bomb it....
  [39] They want me to do a technical presentation. I'm nervous....
  [48] I got the job offer. They want an answer by Friday....


In [8]:
from harness_chandelier.summary import summarize

# English summary — xAI Grok
print("=== Topic Summary (xAI Grok) [EN] ===")
summary_en = summarize(
    result.main_topic_messages,
    provider="grok",
    language="en"
)
print(summary_en)
print()


=== Topic Summary (xAI Grok) [EN] ===


The user is focused on pursuing a career change by applying to and interviewing for a new startup job after feeling stagnant in their current role.

